In [1]:
import numpy as np
import pandas as pd



In [3]:
dataset_path = "../griffin_datasets/airbnb"
griffin_path = "../griffin_datasets/airbnb-pk"

In [7]:
population = pd.read_parquet(f"{dataset_path}/age_gender_bkts.pqt")
# add a primary key
population["primary_key"] = np.arange(len(population))
population


,age_bucket,country_destination,gender,population_in_thousands,year,primary_key
0,100+,AU,male,1.0,2015.0,0
1,95-99,AU,male,9.0,2015.0,1
2,90-94,AU,male,47.0,2015.0,2
3,85-89,AU,male,118.0,2015.0,3
4,80-84,AU,male,199.0,2015.0,4
...,...,...,...,...,...,...
415,95-99,US,male,115.0,2015.0,415
416,90-94,US,male,541.0,2015.0,416
417,15-19,US,female,10570.0,2015.0,417
418,85-89,US,male,1441.0,2015.0,418


In [9]:
session = pd.read_parquet(f"{dataset_path}/sessions.pqt")
session["primary_key"] = np.arange(len(session))
session


,user_id,action,action_type,action_detail,device_type,secs_elapsed,primary_key
0,d1mm9tcy42,lookup,None,None,Windows Desktop,319.0,0
1,d1mm9tcy42,search_results,click,view_search_results,Windows Desktop,67753.0,1
2,d1mm9tcy42,lookup,None,None,Windows Desktop,301.0,2
3,d1mm9tcy42,search_results,click,view_search_results,Windows Desktop,22141.0,3
4,d1mm9tcy42,lookup,None,None,Windows Desktop,435.0,4
...,...,...,...,...,...,...,...
10567732,9uqfg8txu3,dashboard,view,dashboard,Windows Desktop,556.0,10567732
10567733,9uqfg8txu3,edit,view,edit_profile,Windows Desktop,6624.0,10567733
10567734,9uqfg8txu3,webcam_upload,-unknown-,-unknown-,Windows Desktop,200125.0,10567734
10567735,9uqfg8txu3,active,-unknown-,-unknown-,-unknown-,17624.0,10567735


In [10]:
# Save the datasets
import shutil
import os
shutil.rmtree(griffin_path, ignore_errors=True)
shutil.copytree(dataset_path, griffin_path)


'../griffin_datasets/airbnb-pk'

In [11]:
population.to_parquet(f"{griffin_path}/age_gender_bkts.pqt")
session.to_parquet(f"{griffin_path}/sessions.pqt")


dataset_name: airbnb
tables:
  - name: User
    source: users.pqt
    format: parquet
    columns:
      - name: id
        dtype: primary_key
      - name: date_account_created
        dtype: datetime
      - name: timestamp_first_active
        dtype: datetime
      - name: date_first_booking
        dtype: datetime
      - name: gender
        dtype: foreign_key
        link_to: Gender.gender
      - name: age
        dtype: float
      - name: age_bucket
        dtype: foreign_key
        link_to: Age_bucket.age_bucket
      - name: signup_method
        dtype: category
      - name: signup_flow
        dtype: category
      - name: language
        dtype: category
      - name: affiliate_channel
        dtype: category
      - name: affiliate_provider
        dtype: category
      - name: first_affiliate_tracked
        dtype: category
      - name: signup_app
        dtype: category
      - name: first_device_type
        dtype: category
      - name: first_browser
        dtype: category
    time_column: timestamp_first_active
    
  - name: Population
    source: age_gender_bkts.pqt
    format: parquet
    columns:
      - name: primary_key
        dtype: primary_key
      - name: age_bucket
        dtype: foreign_key
        link_to: Age_bucket.age_bucket
      - name: country_destination
        dtype: foreign_key
        link_to: Country.country_destination
      - name: gender
        dtype: foreign_key
        link_to: Gender.gender
      - name: population_in_thousands
        dtype: float
      - name: year
        dtype: datetime
    time_column: year

  - name: Country
    source: countries.pqt
    format: parquet
    columns:
      - name: country_destination
        dtype: primary_key
      - name: lat_destination
        dtype: float
      - name: lng_destination
        dtype: float
      - name: distance_km
        dtype: float
      - name: destination_km2
        dtype: float
      - name: destination_language 
        dtype: category
      - name: language_levenshtein_distance
        dtype: float

  - name: Session
    source: sessions.pqt
    format: parquet
    columns:
      - name: primary_key
        dtype: primary_key
      - name: user_id
        dtype: foreign_key
        link_to: User.id
      - name: action
        dtype: category
      - name: action_type
        dtype: category
      - name: action_detail
        dtype: category
      - name: device_type
        dtype: category
      - name: secs_elapsed
        dtype: float
        
tasks:
  - name: destination
    source: destination/{split}.pqt
    format: parquet
    columns:
      - name: id
        dtype: primary_key
      - name: date_account_created
        dtype: datetime
      - name: timestamp_first_active
        dtype: datetime
      - name: date_first_booking
        dtype: datetime
      - name: gender
        dtype: foreign_key
        link_to: Gender.gender
      - name: age
        dtype: float
      - name: age_bucket
        dtype: foreign_key
        link_to: Age_bucket.age_bucket
      - name: signup_method
        dtype: category
      - name: signup_flow
        dtype: category
      - name: language
        dtype: category
      - name: affiliate_channel
        dtype: category
      - name: affiliate_provider
        dtype: category
      - name: first_affiliate_tracked
        dtype: category
      - name: signup_app
        dtype: category
      - name: first_device_type
        dtype: category
      - name: first_browser
        dtype: category
      - name: country_destination
        dtype: category
    time_column: timestamp_first_active
    evaluation_metric: auroc
    target_column: country_destination
    target_table: User
    task_type: classification
